# Session 5: CNN on MNIST and YOLOv8

This notebook contains structured tasks for building a CNN on MNIST and experimenting with Ultralytics YOLOv8 (pretrained inference and a short training example).

**Instructions:**
- Complete all tasks directly in this notebook.
- Keep your code clear and well-commented.
- Use `%matplotlib inline` for visualizations.
- If you need to install packages (e.g., `ultralytics`), use `%pip` in a cell.
- When using external datasets, do NOT upload them to GitHub.

### ***DO NOT UPLOAD LARGE DATASETS OR MODEL WEIGHTS TO GITHUB***
<hr>

## Task 1: Convolutional Neural Network on MNIST — Build, Train, Evaluate, Visualize
Train a simple CNN on the MNIST handwritten digits dataset. After training, generate predictions on the test set, compare them to ground truth labels, and visualize a few results.

**Why this matters:**
- MNIST is a classic entry point for image classification and CNNs.
- You will practice the full workflow: data loading, model definition, training, evaluation, and visualization.

**Steps:**
- Load MNIST with appropriate transforms (normalize).
- Define a small CNN (Conv → ReLU → Pool → FC).
- Train for a few epochs (2–3 for demo).
- Evaluate on the test set and compute accuracy.
- Visualize several test images with predicted and true labels.

**Hint:** Use `torchvision.datasets.MNIST`, `DataLoader`, `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d`, and `nn.CrossEntropyLoss`.

In [ ]:
# Setup environment and imports
%matplotlib inline

import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Make the results more reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Use the GPU when one is available; otherwise use the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

In [ ]:
# Load MNIST and create DataLoaders.
# MNIST is downloaded automatically the first time this cell runs.
# The standard MNIST mean and standard deviation are used for normalization.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

data_root = "./data"

try:
    train_dataset = datasets.MNIST(
        root=data_root, train=True, transform=transform, download=True
    )
    test_dataset = datasets.MNIST(
        root=data_root, train=False, transform=transform, download=True
    )
except Exception as exc:
    raise RuntimeError(
        "MNIST could not be downloaded. Check your internet connection, "
        "then run this cell again. The dataset is not included in this notebook."
    ) from exc

BATCH_SIZE = 128
# num_workers=0 is compatible with Windows, macOS, Linux, and notebook environments.
pin_memory = (device.type == "cuda")
loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
    generator=loader_generator
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory
)

print(f"Training images: {len(train_dataset):,}")
print(f"Test images: {len(test_dataset):,}")
print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Define a CNN for 1 x 28 x 28 MNIST images.
class MNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # 28 x 28 -> 14 x 14

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)   # 14 x 14 -> 7 x 7
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.25),
            nn.Linear(128, 10)  # Ten digit classes: 0 through 9
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)  # Return raw logits for CrossEntropyLoss


model = MNIST_CNN().to(device)
print(model)

# Confirm that the model returns 10 class scores for each image.
with torch.no_grad():
    sample_logits = model(torch.zeros(2, 1, 28, 28, device=device))
print("Output shape:", tuple(sample_logits.shape))

In [ ]:
# Set up the loss function and optimizer.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train for two epochs (a short demonstration).
NUM_EPOCHS = 2

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Clear old gradients, make predictions, calculate loss, and backpropagate.
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = 100.0 * correct / total
    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"- Loss: {epoch_loss:.4f} "
        f"- Training accuracy: {epoch_accuracy:.2f}%"
    )

In [ ]:
# Evaluate the trained model on the test set.
model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

average_test_loss = test_loss / total
test_accuracy = 100.0 * correct / total

print(f"Test loss: {average_test_loss:.4f}")
print(f"Test accuracy: {correct}/{total} = {test_accuracy:.2f}%")

In [ ]:
# Display a few test images with the predicted and true labels.
model.eval()
images, labels = next(iter(test_loader))
images_device = images.to(device)

with torch.inference_mode():
    logits = model(images_device)
    predicted_labels = logits.argmax(dim=1).cpu()

# Undo normalization only for display.
mean = torch.tensor(0.1307).view(1, 1, 1, 1)
std = torch.tensor(0.3081).view(1, 1, 1, 1)
display_images = (images.cpu() * std + mean).clamp(0, 1)

num_images = min(8, len(images))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes = axes.flatten()

for index in range(num_images):
    axes[index].imshow(display_images[index].squeeze(), cmap="gray")
    pred = predicted_labels[index].item()
    true = labels[index].item()
    axes[index].set_title(f"Predicted: {pred} | True: {true}")
    axes[index].axis("off")

for index in range(num_images, len(axes)):
    axes[index].axis("off")

plt.tight_layout()
plt.show()

## Task 2: YOLOv8 Pretrained Predictions (No Training)
Use a pretrained YOLOv8 model to run inference on any image of your choice and visualize detections.

**Why this matters:**
- Demonstrates transfer learning and using pretrained models for object detection.
- Learn how to quickly test a detector on real images.

**Steps:**
- Install `ultralytics` if not available.
- Load a YOLOv8 model (e.g., `yolov8n.pt`).
- Run predictions on an image path.
- Visualize the detections (boxes, labels, confidences).

**Hint:** Use `from ultralytics import YOLO`, `YOLO('yolov8n.pt')`, and `model.predict(source=...)`.

In [ ]:
# Install Ultralytics only when it is not already available.
import importlib.util

if importlib.util.find_spec("ultralytics") is None:
    %pip install -q ultralytics
else:
    print("Ultralytics is already installed.")

In [ ]:
# Load a pretrained YOLOv8 nano model and run inference.
from ultralytics import YOLO
from ultralytics.utils import ASSETS
import cv2
import matplotlib.pyplot as plt

# Ultralytics ships a small example image (bus.jpg) with its package.
# The pretrained weights are downloaded automatically on the first run.
pretrained_yolo = YOLO("yolov8n.pt")
inference_image = str(ASSETS / "bus.jpg")

prediction_results = pretrained_yolo.predict(
    source=inference_image,
    conf=0.25,
    save=False,
    verbose=False
)

# result.plot() returns a BGR image, so convert it to RGB for Matplotlib.
result = prediction_results[0]
annotated_image = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 7))
plt.imshow(annotated_image)
plt.title("YOLOv8 pretrained detections")
plt.axis("off")
plt.show()

# Print the detected class names and confidence scores.
if result.boxes is not None and len(result.boxes) > 0:
    for box in result.boxes:
        class_id = int(box.cls.item())
        confidence = float(box.conf.item())
        print(f"{result.names[class_id]}: {confidence:.2%}")
else:
    print("No objects were detected in this image.")

## Task 3: YOLOv8 Training Example (Small Demo)
Choose any small dataset (e.g., from Roboflow or Kaggle) and train a YOLOv8 model for a few epochs (e.g., 3) as a demonstration. Then run predictions with the trained model.

**Why this matters:**
- Understand the end-to-end YOLO training pipeline.
- See how dataset configuration (classes, paths) connects to training code.

**Steps:**
- Prepare your dataset directory and a `data.yaml` describing train/val paths and class names.
- Train a small model (e.g., `yolov8n.pt`) for ~3 epochs.
- Run predictions on a few sample images.
- Reflect: Are the predictions what you expected? Why or why not?

**Notes:**
- Do NOT upload datasets to GitHub. Keep them locally or use a data hosting service.
- Training for 3 epochs is just for demonstration; try longer training later.
- If you can't use GPU, small images and fewer epochs help.
- If you can't use GPU, you can train your models on Kaggle or Google Colab in the future for free access to GPUs on their servers

In [ ]:
# Use Ultralytics' small COCO8 dataset for this short training demonstration.
# Ultralytics downloads the dataset automatically when training starts.
# It contains a small number of images and is suitable for testing the pipeline.
data_yaml = "coco8.yaml"

print(f"Dataset configuration: {data_yaml}")
print("The COCO8 dataset will be downloaded automatically if it is not present.")

In [ ]:
# Train a YOLOv8 nano detector for three epochs.
# A GPU is selected automatically when available; CPU is used otherwise.
from pathlib import Path
import torch
from ultralytics import YOLO

training_device = 0 if torch.cuda.is_available() else "cpu"
yolo_training_model = YOLO("yolov8n.pt")

train_results = yolo_training_model.train(
    data=data_yaml,
    epochs=3,
    imgsz=416,
    batch=8,
    device=training_device,
    workers=0,                 # Reliable in notebook and Windows environments
    project="runs/detect",
    name="session5_coco8_demo",
    exist_ok=True,
    plots=True,
    verbose=True
)

# Ultralytics stores the best checkpoint in the trainer's best attribute.
best_weights = Path(yolo_training_model.trainer.best)
print(f"Best model weights: {best_weights}")
print(f"Checkpoint exists: {best_weights.is_file()}")

In [ ]:
# Load the best checkpoint and predict on validation images.
from pathlib import Path
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
import cv2
import matplotlib.pyplot as plt

if not best_weights.is_file():
    raise FileNotFoundError(
        f"Could not find the trained checkpoint at {best_weights}. "
        "Run the training cell above successfully first."
    )

trained_yolo = YOLO(str(best_weights))

# Resolve the validation image path from the dataset YAML.
dataset_info = check_det_dataset(data_yaml)
dataset_root = Path(dataset_info["path"])
validation_entry = dataset_info["val"]

if isinstance(validation_entry, (list, tuple)):
    validation_paths = []
    for entry in validation_entry:
        entry_path = Path(entry)
        if not entry_path.is_absolute():
            entry_path = dataset_root / entry_path
        if entry_path.is_dir():
            validation_paths.extend(
                p for p in entry_path.rglob("*")
                if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
            )
        elif entry_path.is_file():
            validation_paths.append(entry_path)
else:
    validation_path = Path(validation_entry)
    if not validation_path.is_absolute():
        validation_path = dataset_root / validation_path
    if validation_path.is_dir():
        validation_paths = sorted(
            p for p in validation_path.rglob("*")
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
        )
    elif validation_path.is_file():
        validation_paths = [validation_path]
    else:
        validation_paths = []

validation_paths = sorted(validation_paths)
if not validation_paths:
    raise FileNotFoundError(
        "No validation images were found. Check the dataset paths in data_yaml."
    )

sample_sources = [str(path) for path in validation_paths[:3]]
trained_predictions = trained_yolo.predict(
    source=sample_sources,
    conf=0.15,
    save=False,
    verbose=False
)

fig, axes = plt.subplots(
    1, len(trained_predictions),
    figsize=(6 * len(trained_predictions), 5),
    squeeze=False
)
for ax, prediction in zip(axes[0], trained_predictions):
    displayed = cv2.cvtColor(prediction.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(displayed)
    ax.set_title(Path(prediction.path).name)
    ax.axis("off")

plt.tight_layout()
plt.show()

### Reflection

**1. Are the predictions what you expected?**

The pretrained YOLOv8 model should detect common objects that belong to the COCO classes in the example image. The model trained on COCO8 may detect some objects, but its results are expected to be less reliable because it is trained for only three epochs on a very small dataset.

**2. If performance is weak, why?**

- COCO8 contains very few training and validation images.
- Three epochs are not enough for a reliable detector to learn all object variations.
- Small objects, unusual viewpoints, lighting, and occlusion can make detection harder.
- The pretrained model may already know the COCO classes, but the short fine-tuning run provides limited new learning.

**3. What would you adjust for a more serious training run?**

Use a larger, diverse, correctly labelled dataset; train for more epochs; inspect the validation images and labels; tune the image size and batch size; and evaluate precision, recall, and mean Average Precision (mAP). Keep a separate test set for the final evaluation.

**4. What other arguments can you use for the training function?**

Examples include `epochs`, `batch`, `imgsz`, `device`, `workers`, `optimizer`, `lr0` (initial learning rate), `patience` (early stopping), `project`, `name`, `pretrained`, `augment`, `mosaic`, `cache`, `val`, and `plots`. These options control training duration, hardware, optimization, augmentation, validation, and output saving.

**Note:** The notebook uses Ultralytics' small `coco8.yaml` dataset so the training example is reproducible without manually preparing a dataset. The dataset and pretrained weights are downloaded on first use, so an internet connection is required.